In [5]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error

# PyTorch Forecasting + Lightning
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer, MultiNormalizer, TorchNormalizer
from pytorch_forecasting.metrics import SMAPE, RMSE, MAE, QuantileLoss
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.callbacks.progress import TQDMProgressBar
from lightning.pytorch.loggers import TensorBoardLogger

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    torch.set_float32_matmul_precision("medium")  # utilise RTX Tensor Cores
    print("float32 matmul precision set to 'medium'")


# import pytorch_forecasting.models.base._base_model as _bm
# _orig_plot = _bm.BaseModel.log_prediction
#
# def _patched_log_prediction(self, x, out, batch_idx, **kwargs):
#     # cast any bf16 tensors in out to float32 before matplotlib touches them
#     out = {
#         k: v.float() if isinstance(v, torch.Tensor) and v.dtype == torch.bfloat16 else v
#         for k, v in out.items()
#     }
#     return _orig_plot(self, x, out, batch_idx, **kwargs)
#
# _bm.BaseModel.log_prediction = _patched_log_prediction
# ==============================================================================
# Cell 2 – Memory-Optimization Helpers  (unchanged from original script)
# ==============================================================================
def smallest_int_dtype(min_val: int, max_val: int, signed: bool = True) -> str:
    if signed:
        if np.iinfo(np.int8).min <= min_val <= max_val <= np.iinfo(np.int8).max:
            return "int8"
        if np.iinfo(np.int16).min <= min_val <= max_val <= np.iinfo(np.int16).max:
            return "int16"
        if np.iinfo(np.int32).min <= min_val <= max_val <= np.iinfo(np.int32).max:
            return "int32"
        return "int64"
    else:
        if 0 <= min_val <= max_val <= np.iinfo(np.uint8).max:
            return "uint8"
        if 0 <= min_val <= max_val <= np.iinfo(np.uint16).max:
            return "uint16"
        if 0 <= min_val <= max_val <= np.iinfo(np.uint32).max:
            return "uint32"
        return "uint64"


def optimize_df_for_memory(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Converts columns to smaller dtypes.
    For low-decimal float columns, stores scaled integers if that beats float32.
    Returns:
        optimized_df
        metadata dict with scaling info
    """
    meta = {}

    for col in df.columns:
        s = df[col]

        unique_non_null = set(s.dropna().unique())
        if unique_non_null.issubset({0, 1, True, False}):
            if col == "Група":
                df[col] = s.astype("bool")
                meta[col] = {"stored_as": "bool", "scale": 1}
                continue

        if pd.api.types.is_integer_dtype(s):
            mn, mx = int(s.min()), int(s.max())
            dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
            df[col] = s.astype(dtype)
            meta[col] = {"stored_as": dtype, "scale": 1}
            continue

        if pd.api.types.is_float_dtype(s):
            non_null = s.dropna()
            if len(non_null) == 0:
                df[col] = s.astype("float32")
                meta[col] = {"stored_as": "float32", "scale": 1}
                continue

            decimals = non_null.astype(str).apply(
                lambda x: len(x.split(".")[1].rstrip("0")) if "." in x else 0
            ).max()

            if decimals <= 3:
                scale = 10 ** decimals
                scaled = np.round(s * scale)
                mn = int(np.nanmin(scaled))
                mx = int(np.nanmax(scaled))
                int_dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
                int_bytes = np.dtype(int_dtype).itemsize
                float32_bytes = np.dtype("float32").itemsize
                if int_bytes < float32_bytes:
                    df[col] = scaled.astype(int_dtype)
                    meta[col] = {"stored_as": int_dtype, "scale": scale}
                else:
                    df[col] = s.astype("float32")
                    meta[col] = {"stored_as": "float32", "scale": 1}
            else:
                df[col] = s.astype("float32")
                meta[col] = {"stored_as": "float32", "scale": 1}

    return df, meta


# ==============================================================================
# Cell 3 – Metric Helpers  (unchanged from original script)
# ==============================================================================
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8))


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


# ==============================================================================
# Cell 4 – Configuration  ← adjust these before running
# ==============================================================================

# ── Paths ──────────────────────────────────────────────────────────────────────
TRAIN_PATH = "../../../data/money_calc/train.parquet"
VAL_PATH = "../../../data/money_calc/val.parquet"
TEST_PATH = "../../../data/money_calc/test.parquet"

# ── Target ─────────────────────────────────────────────────────────────────────
Y_COL = "Money_spent"

# ── Station identifier column ──────────────────────────────────────────────────
# PyTorch Forecasting needs a single string column that identifies each time series.
# We will derive it from the one-hot EIC columns below (see Cell 5).
GROUP_COL = "station_id"

# ── Forecasting horizon & encoder length ───────────────────────────────────────
# Predict this many hours ahead
MAX_PREDICTION_LENGTH = 24  # 24 h forecast
MAX_ENCODER_LENGTH = 36  # look-back window: 7 days

# ── Training hyper-parameters ──────────────────────────────────────────────────
BATCH_SIZE = 1024 * 2
MAX_EPOCHS = 50
LEARNING_RATE = 3e-3
HIDDEN_SIZE = 32  # was 64
ATTENTION_HEAD_SIZE = 2  # was 4
HIDDEN_CONTINUOUS_SIZE = 16  # was 32
DROPOUT = 0.1

# ── Numerical real-valued covariates known at forecast time (time-varying) ─────
FUTURE_REALS = [
    "temperature_2m", "apparent_temperature", "dew_point_2m",
    "relative_humidity_2m", "precipitation", "rain", "snowfall",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "surface_pressure", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m",
    "shortwave_radiation", "diffuse_radiation", "direct_normal_irradiance",
    # removed: Hour, day_of_week, season_number, Month, Day
    # these are now handled by TIME_VARYING_KNOWN_CATS below
    "Average of Ціна розподілу ЕЕ грн_ без ПДВ/кВт*год",
    "Average of Ціна ЕЕ грн_ без ПДВ/кВт*год",
]

# ── Numerical real-valued covariates NOT known at forecast time ─────────────────
PAST_REALS = []  # add any columns that are only available historically

# ── Static numerical covariates (one value per station, time-invariant) ─────────
STATIC_REALS = [
    "GPS-координати - Широта",
    "GPS-координати - Довгота",
]

# ── Categorical time-varying known covariates ───────────────────────────────────
# Calendar categoricals — always known in the future
TIME_VARYING_KNOWN_CATS = [
    "Month_cat", "Day_cat", "Hour_cat", "day_of_week_cat", "season_cat"
]  # created in Cell 5

# ── Static categoricals ─────────────────────────────────────────────────────────
# One-hot encoded columns will be collapsed back to a single categorical.
# Any remaining OHE groups (Тип, Область, …) are handled similarly.
STATIC_CATS = ["station_id", "Тип_cat", "Область_cat"]  # created in Cell 5


# ==============================================================================
# Cell 5 – Load & Pre-process Data
# ==============================================================================

def recover_station_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    The 400 'EIC-код_<station>' columns are one-hot encoded.
    Collapse them into a single station_id string column.
    """
    eic_cols = [c for c in df.columns if c.startswith("EIC-код_")]
    # argmax across OHE columns → station name
    df[GROUP_COL] = df[eic_cols].idxmax(axis=1).str.replace("EIC-код_", "", regex=False)
    df = df.drop(columns=eic_cols)
    return df


def recover_ohe_cat(df: pd.DataFrame, prefix: str, new_col: str) -> pd.DataFrame:
    """
    Collapse a set of OHE columns back to a single categorical column.
    prefix  – common column prefix, e.g. 'Тип_'
    new_col – name for the resulting categorical column
    """
    ohe_cols = [c for c in df.columns if c.startswith(prefix)]
    if not ohe_cols:
        return df
    df[new_col] = df[ohe_cols].idxmax(axis=1).str.replace(prefix, "", regex=False)
    df = df.drop(columns=ohe_cols)
    return df


def build_time_index(df: pd.DataFrame) -> pd.DataFrame:
    """
    PyTorch Forecasting requires a monotonically increasing integer time_idx
    that is *shared* across all groups (think of it as the global hour counter).
    """
    df["datetime"] = pd.to_datetime(
        df[["Year", "Month", "Day", "Hour"]].rename(
            columns={"Year": "year", "Month": "month", "Day": "day", "Hour": "hour"}
        )
    )
    # Global hourly index starting from the minimum timestamp in the dataset
    min_dt = df["datetime"].min()
    df["time_idx"] = ((df["datetime"] - min_dt).dt.total_seconds() / 3600).astype(int)
    return df


def add_cat_helpers(df: pd.DataFrame) -> pd.DataFrame:
    """Create string categoricals from numerics for PyTorch Forecasting."""
    df["Month_cat"] = df["Month"].astype(str)
    df["day_of_week_cat"] = df["day_of_week"].astype(str)
    df["season_cat"] = df["season_number"].astype(str)
    return df


DROP_PREFIXES = ("АЗС_", "ОСР код_", "ОСР опис_")


def drop_prefix_cols(df: pd.DataFrame) -> pd.DataFrame:
    cols_to_drop = [c for c in df.columns if c.startswith(DROP_PREFIXES)]
    print(f"  Dropping {len(cols_to_drop)} prefix-matched columns")
    return df.drop(columns=cols_to_drop)


def add_cat_helpers(df: pd.DataFrame) -> pd.DataFrame:
    """Create string categoricals from numerics for PyTorch Forecasting."""
    df["Month_cat"] = df["Month"].astype(str)
    df["Day_cat"] = df["Day"].astype(str)
    df["Hour_cat"] = df["Hour"].astype(str)
    df["day_of_week_cat"] = df["day_of_week"].astype(str)
    df["season_cat"] = df["season_number"].astype(str)
    return df


def load_and_prepare(path: str) -> pd.DataFrame:
    df = pd.read_parquet(path).reset_index(drop=True)

    df.columns = df.columns.str.replace('.', '_', regex=False)
    df, _ = optimize_df_for_memory(df)

    # Cast compressed ints back to float32 for PTF
    for col in df.select_dtypes(
            include=["int8", "int16", "int32", "uint8", "uint16", "uint32"]
    ).columns:
        if col not in ["Year", "Month", "Day", "Hour", "day_of_week", "season_number"]:
            df[col] = df[col].astype("float32")

    # Drop junk OHE columns before anything else
    df = drop_prefix_cols(df)

    df = recover_station_id(df)
    df = recover_ohe_cat(df, prefix="Тип_", new_col="Тип_cat")
    df = recover_ohe_cat(df, prefix="Область_", new_col="Область_cat")
    df = build_time_index(df)
    df = add_cat_helpers(df)

    # # Filter to data from 2024-09-01 onwards
    df = df[df["datetime"] >= "2024-07-10"].reset_index(drop=True)

    df[Y_COL] = df[Y_COL].astype("float32")
    df = df.sort_values([GROUP_COL, "time_idx"]).reset_index(drop=True)

    return df


print("Loading train …")
train = load_and_prepare(TRAIN_PATH)

# Compute global min datetime from train so val/test share the same time_idx origin
GLOBAL_MIN_DT = train["datetime"].min()

print("Loading val   …")
val = load_and_prepare(VAL_PATH)

print("Loading test  …")
test = load_and_prepare(TEST_PATH)

# Align time_idx to the same global origin across splits
for split in [val, test]:
    split["time_idx"] = ((split["datetime"] - GLOBAL_MIN_DT).dt.total_seconds() / 3600).astype(int)

print(f"\nTrain shape : {train.shape}")
print(f"Val   shape : {val.shape}")
print(f"Test  shape : {test.shape}")
print(f"\nStations in train : {train[GROUP_COL].nunique()}")
print(f"time_idx range    : [{train['time_idx'].min()}, {train['time_idx'].max()}]")
# Station diagnostics — run before deciding which to cut
station_stats = (
    train.groupby(GROUP_COL)
    .agg(
        rows=(Y_COL, "count"),
        hours_span=("time_idx", lambda x: x.max() - x.min()),
        missing_pct=("time_idx", lambda x: 1 - len(x) / (x.max() - x.min() + 1)),
        target_mean=(Y_COL, "mean"),
        target_std=(Y_COL, "std"),
        target_zeros=(Y_COL, lambda x: (x == 0).mean()),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

print(f"Total stations: {len(station_stats)}")
print(f"\nTop 10 largest:")
print(station_stats.head(10).to_string())
print(f"\nBottom 10 smallest:")
print(station_stats.tail(10).to_string())
print(f"\nStations with >50% missing: {(station_stats.missing_pct > 0.5).sum()}")
print(f"Stations with >50% zeros  : {(station_stats.target_zeros > 0.5).sum()}")
print(f"Stations with <100 rows   : {(station_stats.rows < 100).sum()}")
TARGET_STATIONS = 100

np.random.seed(42)

sampled_stations = station_stats.sample(
    n=TARGET_STATIONS, random_state=42
)[GROUP_COL].values

print(f"Stations kept: {len(sampled_stations)}")

train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Train rows     : {len(train):,}")
print(f"Train batches  : {len(train) // BATCH_SIZE}")

PyTorch version  : 2.6.0+cu124
CUDA available   : True
GPU              : NVIDIA GeForce RTX 3090
float32 matmul precision set to 'medium'
Loading train …
  Dropping 157 prefix-matched columns
Loading val   …
  Dropping 157 prefix-matched columns
Loading test  …
  Dropping 157 prefix-matched columns

Train shape : (3224230, 40)
Val   shape : (304499, 40)
Test  shape : (304152, 40)

Stations in train : 408
time_idx range    : [4583, 13126]
Total stations: 408

Top 10 largest:
           station_id  rows  hours_span  missing_pct  target_mean  target_std  target_zeros
0    62Z0008583037334  8543        8543     0.000117   172.163116   43.034962      0.000000
259  62Z6070434707477  8543        8543     0.000117   157.028549   38.818211      0.000000
269  62Z6337618908569  8543        8543     0.000117    90.108482   29.581688      0.000000
268  62Z6308545383007  8543        8543     0.000117    87.902588   39.339725      0.000000
267  62Z6295419983830  8543        8543     0.000117   116.2

In [12]:
# ==============================================================================
# LightGBM Benchmark – Data Preparation
# ==============================================================================
import lightgbm as lgb

DROP_FOR_LGB = [
    "datetime",
    "time_idx",
    "Month_cat", "Day_cat", "Hour_cat", "day_of_week_cat", "season_cat",
]

CAT_COLS = ["station_id", "Тип_cat", "Область_cat"]

# Fit OHE on train only, apply to all splits
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore",  # unseen categories in val/test become all zeros
    dtype=np.float32,
)
ohe.fit(train[CAT_COLS])

def prepare_lgb(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop(columns=[c for c in DROP_FOR_LGB if c in df.columns])

    # One-hot encode categoricals
    ohe_array = ohe.transform(df[CAT_COLS])
    ohe_cols  = ohe.get_feature_names_out(CAT_COLS)
    ohe_df    = pd.DataFrame(ohe_array, columns=ohe_cols, index=df.index)

    df = df.drop(columns=CAT_COLS)
    df = pd.concat([df, ohe_df], axis=1)
    return df

train_lgb = prepare_lgb(train.copy())
val   = prepare_lgb(val.copy())
test  = prepare_lgb(test.copy())

features = [c for c in train_lgb.columns if c != Y_COL]

X_train = train_lgb[features]
y_train = train_lgb[Y_COL]

print(f"Features : {len(features)}")
print(f"Train rows: {len(X_train):,}")

Features : 156
Train rows: 781,932


In [8]:
X_train

,Year,Month,Day,Hour,Average of Ціна розподілу ЕЕ грн_ без ПДВ/кВт*год,Average of Ціна ЕЕ грн_ без ПДВ/кВт*год,GPS-координати - Широта,GPS-координати - Довгота,temperature_2m,apparent_temperature,...,Область_cat_Одеська,Область_cat_Полтавська,Область_cat_Рівненська,Область_cat_Тернопільська,Область_cat_Харківська,Область_cat_Херсонська,Область_cat_Хмельницька,Область_cat_Черкаська,Область_cat_Чернігівська,Область_cat_м_ Київ
0,2024,7,9,24,2.15221,5.732343,48.442429,22.192190,234.0,268.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2024,7,10,1,2.15221,5.732343,48.442429,22.192190,225.0,259.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2024,7,10,2,2.15221,5.732343,48.442429,22.192190,218.0,246.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024,7,10,3,2.15221,5.732343,48.442429,22.192190,217.0,246.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2024,7,10,4,2.15221,5.732343,48.442429,22.192190,217.0,243.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
781927,2025,6,30,19,1.70713,4.883898,50.419476,29.842367,180.0,139.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
781928,2025,6,30,20,1.70713,4.883898,50.419476,29.842367,146.0,121.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
781929,2025,6,30,21,1.70713,4.883898,50.419476,29.842367,148.0,113.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
781930,2025,6,30,22,1.70713,4.883898,50.419476,29.842367,148.0,119.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
lgb_model = lgb.LGBMRegressor()
lgb_model.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016786 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3800
[LightGBM] [Info] Number of data points in the train set: 781932, number of used features: 156
[LightGBM] [Info] Start training from score 129.554425


LGBMRegressor()

In [14]:
val_pred = lgb_model.predict(val[features])
test_pred = lgb_model.predict(test[features])

In [15]:
print("\nValidation metrics")
print("SMAPE:", smape(val[Y_COL], val_pred))
print("RMSE :", rmse(val[Y_COL], val_pred))
print("MAPE :", mape(val[Y_COL], val_pred))

print("\nTest metrics")
print("SMAPE:", smape(test[Y_COL], test_pred))
print("RMSE :", rmse(test[Y_COL], test_pred))
print("MAPE :", mape(test[Y_COL], test_pred))


Validation metrics
SMAPE: 0.20544847638107033
RMSE : 46.68033164198164
MAPE : 23.293860795481987

Test metrics
SMAPE: 0.20535744368550876
RMSE : 47.52214576188987
MAPE : 39.91615039493421
